# Introspection Experiments — Colab Runner

This notebook runs the full introspection experiment pipeline on Colab GPUs.

**Pipeline:**
1. Setup & install dependencies
2. Generate steering vectors (or use pre-computed ones)
3. Run intervention experiments (control + steering)
4. Download results

**Requirements:** Colab Pro+ with A100 GPU recommended.

## 1. Check GPU & Setup

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Clone repo and install dependencies
# Note: Colab doesn't have Python 3.13 or uv, so we install deps directly
# and add src/ to the path instead of doing `pip install -e .`
!git lfs install
!git clone https://github.com/agastyasridharan/introspection.git
%cd introspection

# Install deps (torch is already pre-installed on Colab, so skip it)
!pip install -q accelerate transformers huggingface-hub tqdm 2>&1 | tail -3

# Add src/ to Python path so `from introspection import ...` works
import sys
sys.path.insert(0, "src")
print(f"Python {sys.version}")
print("Setup complete")

In [ ]:
# Verify the package imports correctly
from introspection import generate_steering_vectors, steer, hooks
print("Package imported successfully")

## 2. Configuration

Choose the model and experiment parameters. The GPU VRAM determines which models are feasible:

| Model | VRAM (float16) | GPU Required |
|-------|---------------|-------------|
| Qwen3-8B | ~16GB | T4 / A100 |
| Qwen3-14B | ~28GB | A100 40GB |
| Qwen3-32B | ~64GB | A100 80GB |

In [ ]:
# ===== EDIT THESE =====

MODEL_NAME = "Qwen/Qwen3-8B"       # Options: "Qwen/Qwen3-8B", "Qwen/Qwen3-14B", "Qwen/Qwen3-32B"
DTYPE = "bfloat16"                   # "bfloat16" for A100, "float16" for T4/V100
CONCEPT_COUNT = 50                   # Number of concepts (max 50)
SEED = 13                           # Random seed for reproducibility

# Experiment sweep parameters
LAYERS = [5, 10, 15, 20, 25, 30, 35]  # Layer indices to inject at (adjust per model)
STRENGTHS = [3.5, 4.0, 4.5, 5.0, 6.0] # Steering vector multipliers
TEMPERATURES = [0.7]                    # Sampling temperatures
TRIALS = 5                             # Trials per configuration (more = better stats)
MAX_BATCH_SIZE = 10                     # Reduce if OOM

# Derived paths
import re
model_short = re.search(r"(\d+B)", MODEL_NAME).group(1).lower()
DATA_DIR = f"data/qwen_{model_short}"
STEERING_VECTOR_PATH = f"{DATA_DIR}/steering_vectors.pt"
SWEEP_OUTPUT_PATH = f"{DATA_DIR}/sweep_colab.json"

# Layer count validation
from introspection.constants import MODEL_LAYER_COUNTS
total_layers = MODEL_LAYER_COUNTS.get(MODEL_NAME)
if total_layers:
    invalid = [l for l in LAYERS if l >= total_layers]
    if invalid:
        raise ValueError(f"Layers {invalid} exceed model's {total_layers} layers")
    print(f"Model: {MODEL_NAME} ({total_layers} layers)")
else:
    print(f"Model: {MODEL_NAME} (layer count not in constants, verify manually)")

print(f"Layers: {LAYERS}")
print(f"Strengths: {STRENGTHS}")
print(f"Trials: {TRIALS}")
print(f"Output: {SWEEP_OUTPUT_PATH}")

## 3. Generate Steering Vectors

Skip this cell if you already have `steering_vectors.pt` from Git LFS.

In [ ]:
import os
from pathlib import Path

sv_path = Path(STEERING_VECTOR_PATH)
if sv_path.exists() and sv_path.stat().st_size > 1000:
    print(f"Steering vectors already exist at {sv_path} ({sv_path.stat().st_size / 1e6:.1f} MB)")
    print("Skipping generation. Delete the file to regenerate.")
else:
    print(f"Generating steering vectors for {MODEL_NAME}...")
    from introspection.generate_steering_vectors import run_experiment
    run_experiment(
        model_name=MODEL_NAME,
        dtype_name=DTYPE,
        output_path=sv_path,
        concept_count=CONCEPT_COUNT,
        seed=SEED,
    )
    print(f"Done! Saved to {sv_path} ({sv_path.stat().st_size / 1e6:.1f} MB)")

## 4. Run Intervention Experiments

This is the main experiment: for each (concept × layer × strength × trial), generate a control response (no steering) and an intervention response (with steering vector injected).

**Runtime estimate:** ~1-3 hours for 50 concepts × 7 layers × 5 strengths × 5 trials on A100.

In [ ]:
from pathlib import Path
from introspection.steer import (
    load_steering_vectors, prepare_prompt, steer as run_steer,
    PROMPT_MESSAGES,
)
from introspection.types import ExperimentArgs
from introspection.utils import load_model, resolve_torch_dtype
import json

# Load model (reuses if already loaded from step 3)
print(f"Loading {MODEL_NAME}...")
tokenizer, model = load_model(
    model_name=MODEL_NAME,
    dtype=resolve_torch_dtype(DTYPE),
    disable_cache=False,
    set_pad_token_to_eos=True,
)
print(f"Model loaded on {model.device}")

# Load steering vectors
steering_vectors = load_steering_vectors(Path(STEERING_VECTOR_PATH))
concept_names = sorted(steering_vectors.keys())
print(f"Loaded {len(concept_names)} concepts: {concept_names[:5]}...")

# Prepare prompt
template_prompt = prepare_prompt(tokenizer, model.device)

# Build experiment args
args = ExperimentArgs(
    model_name=MODEL_NAME,
    dtype_name=DTYPE,
    steering_vector_path=Path(STEERING_VECTOR_PATH),
    concepts=None,  # use all
    layers=LAYERS,
    strengths=STRENGTHS,
    json_path=Path(SWEEP_OUTPUT_PATH),
    temperatures=TEMPERATURES,
    top_p=0.8,
    top_k=20,
    min_p=0.0,
    trials=TRIALS,
    max_new_tokens=200,
    do_sample=True,
    seed=SEED,
    debug_residual=False,
    max_batch_size=MAX_BATCH_SIZE,
)

# Run!
print(f"\n=== Running {len(concept_names)} concepts × {len(LAYERS)} layers × {len(STRENGTHS)} strengths × {TRIALS} trials ===")
records = run_steer(
    args=args,
    concept_names=concept_names,
    all_steering_vectors=steering_vectors,
    tokenizer=tokenizer,
    model=model,
    template_prompt=template_prompt,
)

# Save results
output_path = Path(SWEEP_OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)
experiment_summary = {
    "model_name": MODEL_NAME,
    "steering_vector_path": str(args.steering_vector_path),
    "dtype": DTYPE,
    "prompt": {
        "messages": PROMPT_MESSAGES,
        "formatted": template_prompt.formatted_prompt,
        "injection_index": template_prompt.injection_index,
    },
    "settings": {
        "strengths": STRENGTHS,
        "temperatures": TEMPERATURES,
        "top_p": args.top_p,
        "top_k": args.top_k,
        "min_p": args.min_p,
        "max_new_tokens": args.max_new_tokens,
        "trials": TRIALS,
        "do_sample": args.do_sample,
        "seed": SEED,
        "layers": LAYERS,
        "concepts_requested": None,
        "max_batch_size": MAX_BATCH_SIZE,
    },
    "concepts_evaluated": concept_names,
    "results": records,
}
with output_path.open("w", encoding="utf-8") as f:
    json.dump(experiment_summary, f, ensure_ascii=False, indent=2)
    f.write("\n")

print(f"\nSaved {len(records)} records to {output_path}")

## 5. Download Results

Download the sweep JSON to your local machine. You can then commit it to your repo and run grading (Stage 3) and visualization (Stage 4) locally.

In [ ]:
from google.colab import files
import os

# Download the sweep results
sweep_path = SWEEP_OUTPUT_PATH
if os.path.exists(sweep_path):
    size_mb = os.path.getsize(sweep_path) / 1e6
    print(f"Downloading {sweep_path} ({size_mb:.1f} MB)...")
    files.download(sweep_path)
else:
    print(f"No file at {sweep_path} — run the experiment first")

# Also download steering vectors if they were freshly generated
sv_path = STEERING_VECTOR_PATH
if os.path.exists(sv_path):
    size_mb = os.path.getsize(sv_path) / 1e6
    print(f"\nSteering vectors at {sv_path} ({size_mb:.1f} MB)")
    # Uncomment to download:
    # files.download(sv_path)

## 6. Quick Sanity Check

Preview a few results before downloading.

In [ ]:
import json, random

with open(SWEEP_OUTPUT_PATH) as f:
    data = json.load(f)

results = data["results"]
print(f"Total records: {len(results)}")
print(f"Concepts: {len(data['concepts_evaluated'])}")
print(f"Layers: {sorted(set(str(r['layers']) for r in results))}")
print(f"Strengths: {sorted(set(r['strength'] for r in results))}")
print(f"Trials: {sorted(set(r['trial'] for r in results))}")

# Show a few random examples
print("\n" + "="*80)
for r in random.sample(results, min(3, len(results))):
    print(f"\nConcept: {r['concept']} | Layer: {r['layers']} | Strength: {r['strength']}")
    print(f"  Control:      {r['control'][:150]}...")
    print(f"  Intervention: {r['intervention'][:150]}...")

---

## 7. Logit-Based Introspection Experiment (NEW)

This is a cleaner version of the experiment that uses **logit extraction** instead of free-form generation + LLM grading. It runs a **2x2 design**: detection vs factual control questions, with and without steering injection.

- **Detection prompt**: "Did you detect an injected thought?" (with steering)
- **Factual control**: "Can humans breathe underwater?" (with same steering)

If the model shows elevated YES-logits for detection but NOT for factual questions, that's evidence for genuine introspection rather than a general YES-bias.

~200x faster than the generation-based experiment (single forward pass per batch).

In [ ]:
# Logit experiment configuration
# Uses the same MODEL_NAME, DTYPE, LAYERS, STRENGTHS, SEED from Section 2 above.
# Reuses model and steering vectors if already loaded from Section 4.

LOGIT_OUTPUT_PATH = f"{DATA_DIR}/logit_experiment.json"
LOGIT_MAX_BATCH_SIZE = 50  # Can be larger than generation since only 1 forward pass

print(f"Model: {MODEL_NAME}")
print(f"Layers: {LAYERS}")
print(f"Strengths: {STRENGTHS}")
print(f"Output: {LOGIT_OUTPUT_PATH}")

In [ ]:
from pathlib import Path
from introspection.steer import load_steering_vectors, set_random_seed
from introspection.logit_steer import run_logit_experiment
from introspection.types import LogitExperimentArgs
from introspection.utils import load_model, resolve_torch_dtype
import json

# Load model if not already loaded
try:
    _ = model.device
    print(f"Reusing model already on {model.device}")
except NameError:
    print(f"Loading {MODEL_NAME}...")
    tokenizer, model = load_model(
        model_name=MODEL_NAME,
        dtype=resolve_torch_dtype(DTYPE),
        disable_cache=False,
        set_pad_token_to_eos=True,
    )
    print(f"Model loaded on {model.device}")

# Load steering vectors if not already loaded
try:
    _ = steering_vectors
    print(f"Reusing {len(steering_vectors)} steering vectors")
except NameError:
    steering_vectors = load_steering_vectors(Path(STEERING_VECTOR_PATH))
    print(f"Loaded {len(steering_vectors)} concepts")

set_random_seed(SEED)

args = LogitExperimentArgs(
    model_name=MODEL_NAME,
    dtype_name=DTYPE,
    steering_vector_path=Path(STEERING_VECTOR_PATH),
    concepts=None,
    layers=LAYERS,
    strengths=STRENGTHS,
    json_path=Path(LOGIT_OUTPUT_PATH),
    seed=SEED,
    debug_residual=False,
    max_batch_size=LOGIT_MAX_BATCH_SIZE,
)

output = run_logit_experiment(
    args=args,
    model=model,
    tokenizer=tokenizer,
    all_steering_vectors=steering_vectors,
)

# Save
output_path = Path(LOGIT_OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
    f.write("\n")

n_detection = sum(1 for r in output["results"] if r["condition"] == "detection")
n_factual = sum(1 for r in output["results"] if r["condition"] == "factual")
print(f"\nSaved {len(output['results'])} records ({n_detection} detection, {n_factual} factual)")
print(f"  to {output_path}")

### Quick Analysis: Detection vs Factual Control

In [ ]:
import json
import numpy as np

with open(LOGIT_OUTPUT_PATH) as f:
    data = json.load(f)

results = data["results"]
detection_baseline = data["baselines"]["detection_no_injection"]["logit_diff"]

# Separate detection and factual results
det = [r for r in results if r["condition"] == "detection"]
fac = [r for r in results if r["condition"] == "factual"]

print(f"Detection baseline (no injection): logit_diff = {detection_baseline:.3f}")
print(f"{'':>20} {'Detection':>12} {'Factual':>12} {'Introspection':>14}")
print("-" * 62)

# Group by (layer, strength) and compute mean logit_diff
from collections import defaultdict
det_by_config = defaultdict(list)
fac_by_config = defaultdict(list)

for r in det:
    det_by_config[(r["layer"], r["strength"])].append(r["logit_diff"])
for r in fac:
    fac_by_config[(r["layer"], r["strength"])].append(r["logit_diff"])

for (layer, strength) in sorted(det_by_config.keys()):
    d_mean = np.mean(det_by_config[(layer, strength)])
    f_mean = np.mean(fac_by_config[(layer, strength)])
    introspection = d_mean - f_mean
    print(f"  L{layer:>3} S{strength:>4.1f}    {d_mean:>+10.3f}   {f_mean:>+10.3f}   {introspection:>+12.3f}")

### Download Logit Experiment Results

In [ ]:
from google.colab import files
import os

if os.path.exists(LOGIT_OUTPUT_PATH):
    size_mb = os.path.getsize(LOGIT_OUTPUT_PATH) / 1e6
    print(f"Downloading {LOGIT_OUTPUT_PATH} ({size_mb:.1f} MB)...")
    files.download(LOGIT_OUTPUT_PATH)
else:
    print(f"No file at {LOGIT_OUTPUT_PATH} — run the logit experiment first")